In [67]:
from collections import Counter
import re
import json
from typing_extensions import override

import pandas as pd
from datasets import load_dataset

from tqdm import tqdm
from constants.config import init_env_and_logger

logger = init_env_and_logger()

# Load the OctoBench
OctoBench_id = "MiniMaxAI/OctoBench"
logger.info("Now Loading the origin OctoBench dataset")
ds = load_dataset(
    "json",
    data_files={
        "train": "hf://datasets/MiniMaxAI/OctoBench/OctoBench.jsonl"
    },
    split = "train"
)
print(ds)


ImportError: cannot import name 'init_env_and_logger' from 'constants.config' (/Users/zpl/code/CodeMem/scripts/constants/config.py)

In [19]:
# Filter for multi-user_query

multi_query_ds = ds.filter(lambda x: len(x["user_query"]) > 1)
print(multi_query_ds)
print([len(x) for x in multi_query_ds["user_query"]])

Dataset({
    features: ['instance_id', 'user_query', 'system_prompt', 'category', 'image', 'workspace_abs_path', 'scaffold', 'checklist', 'expected_skill'],
    num_rows: 14
})
[3, 3, 3, 2, 3, 2, 3, 2, 5, 2, 3, 2, 3, 2]


In [37]:
single_query_ds = ds.filter(lambda x: len(x["user_query"]) == 1)

# System_Prompt feature
sys = [x for x in ds["system_prompt"] if x != ""]
print(len(sys))

# Category Feature
category = set(ds["category"])
print(category)

sp_rows = ds.filter(lambda x: x["category"] == "SP")
print(sp_rows)

mem = ds.filter(lambda x: x["category"] == "memory")
print(mem)

skill = ds.filter(lambda x: x["category"] == "Skill")
print(skill)

claude = ds.filter(lambda x: x["category"] == "Claude.md")
print(claude)

agent = ds.filter(lambda x: x["category"] == "AGENTS.md")
print(agent)

UQ = ds.filter(lambda x: x["category"] == "User Query")
print(UQ)

print(claude[0]["workspace_abs_path"])

55
{'SP', 'memory', 'Skill', 'Claude.md', 'AGENTS.md', 'User Query'}
Dataset({
    features: ['instance_id', 'user_query', 'system_prompt', 'category', 'image', 'workspace_abs_path', 'scaffold', 'checklist', 'expected_skill'],
    num_rows: 55
})
Dataset({
    features: ['instance_id', 'user_query', 'system_prompt', 'category', 'image', 'workspace_abs_path', 'scaffold', 'checklist', 'expected_skill'],
    num_rows: 29
})
Dataset({
    features: ['instance_id', 'user_query', 'system_prompt', 'category', 'image', 'workspace_abs_path', 'scaffold', 'checklist', 'expected_skill'],
    num_rows: 46
})
Dataset({
    features: ['instance_id', 'user_query', 'system_prompt', 'category', 'image', 'workspace_abs_path', 'scaffold', 'checklist', 'expected_skill'],
    num_rows: 35
})
Dataset({
    features: ['instance_id', 'user_query', 'system_prompt', 'category', 'image', 'workspace_abs_path', 'scaffold', 'checklist', 'expected_skill'],
    num_rows: 25
})
Dataset({
    features: ['instance_id', '

In [ ]:
import subprocess
import uuid
import os

def run(cmd):
    return subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

def extract_from_image(image, container_path, local_paths):
    container_name = f"tmp_{uuid.uuid4().hex[:8]}"

    try:
        # 1. Pull the image
        run(["docker", "pull", "--platform=linux/amd64", image])

        # 2. Create the container
        run(["docker", "create", "--name", container_name, image])

        # 3. Copy File
        for local_path in local_paths:
            try:
                run([
                    "docker", "cp",
                    f"{container_name}:{container_path}",
                    local_path
                ])
            except Exception as e:
                print(f"[ERROR] {image}_{container_name}_{container_path}: copy file error[{str(e)}]")
                continue

    except subprocess.CalledProcessError as e:
        print(f"[ERROR] {image}: {e.stderr.decode()}")
        return False

    finally:
        # 4. Delete Container
        subprocess.run(["docker", "rm", "-f", container_name],
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL)

        # 5. Delete image
        subprocess.run(["docker", "rmi", "-f", image],
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL)

    return True

def docker_login(username, token):
        subprocess.run(
            ["docker", "login", "-u", username, "--password-stdin"],
            input=token.encode(),
            check=True
        )


def deal_one_instance(instance):
    """ 
    instance features:
    features: ['instance_id', 'user_query', 'system_prompt', 'category', 'image', 'workspace_abs_path', 'scaffold', 'checklist', 'expected_skill']

    Output: instance_query_dict:
    {
        "user_query": [xxx],
        "system_prompt": xxx,
        "Claude.md": xxx,
        "AGENT.md": xxx
        # Skill and Memory File is hard to extract, so not considered yet
        # Or maybe Skill can be downloaded ?
    }
    """

    qd = {
        "user_query" : instance["user_query"],
        "system_prompt" : instance["system_prompt"],
        "Claude.md" : "",
        "AGENTS.md" : ""
    }
    LOCAL_ADDR = f"../data/octo_image_file/{instance["instance_id"]}"
    os.makedirs(LOCAL_ADDR, exist_ok = True)


    CLAUDE_ADDR = LOCAL_ADDR + "/Claude.md"
    AGENTS_ADDR = LOCAL_ADDR + "/AGENTS.md"
    extract_from_image(instance["image"], instance["workspace_abs_path"] + "/CLAUDE.md", [CLAUDE_ADDR, AGENTS_ADDR])

    try:
        with open(CLAUDE_ADDR, "r", encoding="utf-8") as f:
            claude_str = f.read()
            qd["Claude.md"] = claude_str
    except FileNotFoundError :
        logger.warning(f"Instance {instance["instance_id"]}: Claude.md not found")

    try:
        with open(AGENTS_ADDR, "r", encoding="utf-8") as f:
            agents_str = f.read()
            qd["AGENTS.md"] = agents_str
    except FileNotFoundError :
        logger.warning(f"Instance {instance["instance_id"]}: AGENTS.md not found")

    return qd


username = os.getenv("DOCKER_USERNAME")
token = os.getenv("DOCKER_TOKEN")
# print(username)
# print(token)

logger.info("Try login docker")
try:
    docker_login(username, token)
except Exception as e:
    logger.error(f"Docker login error:{str(e)}")

ds_without_mem_or_skill = ds.filter(lambda x: x["category"] not in ["memory", "Skill"])
instance_query_dict = {}
for instance in tqdm(ds_without_mem_or_skill):
    instance_query_dict[instance["instance_id"]] = deal_one_instance(instance)

from constants.addr import INSTANCE_QUERY_ADDR
with open(INSTANCE_QUERY_ADDR, "w", encoding="utf-8") as f:
    json.dump(instance_query_dict, f)


2026-07-06 14:31:26,746 - INFO - Try login docker


Login Succeeded


  1%|          | 1/142 [00:03<08:32,  3.64s/it]

[ERROR] minimaxai/feedfeed:md_inkline_tmp_23b26164_/workspace/inkline/CLAUDE.md: copy file error[Command '['docker', 'cp', 'tmp_23b26164:/workspace/inkline/CLAUDE.md', '../data/octo_image_file/agents-inkline-type-guard/Claude.md']' returned non-zero exit status 1.]
[ERROR] minimaxai/feedfeed:md_inkline_tmp_23b26164_/workspace/inkline/CLAUDE.md: copy file error[Command '['docker', 'cp', 'tmp_23b26164:/workspace/inkline/CLAUDE.md', '../data/octo_image_file/agents-inkline-type-guard/AGENTS.md']' returned non-zero exit status 1.]


2026-07-06 14:33:43,160 - WARNING - Instance agents-inkline-type-guard: Claude.md not found
2026-07-06 14:33:43,161 - WARNING - Instance agents-inkline-type-guard: AGENTS.md not found
  1%|▏         | 2/142 [02:11<2:58:44, 76.60s/it]2026-07-06 14:33:55,805 - WARNING - Instance md-basic-memory-async-client-pattern: Claude.md not found
2026-07-06 14:33:55,806 - WARNING - Instance md-basic-memory-async-client-pattern: AGENTS.md not found
  2%|▏         | 3/142 [02:23<1:49:48, 47.40s/it]

...

2026-07-06 20:23:24,746 - WARNING - Instance md-jsbeeb-gamepad-handler: Claude.md not found
2026-07-06 20:23:24,750 - WARNING - Instance md-jsbeeb-gamepad-handler: AGENTS.md not found
 99%|█████████▊| 140/142 [5:51:52<06:10, 185.17s/it]

[ERROR] minimaxai/feedfeed:md_inkline_tmp_630ddab0_/workspace/inkline/CLAUDE.md: copy file error[Command '['docker', 'cp', 'tmp_630ddab0:/workspace/inkline/CLAUDE.md', '../data/octo_image_file/md-inkline-use-notification/Claude.md']' returned non-zero exit status 1.]
[ERROR] minimaxai/feedfeed:md_inkline_tmp_630ddab0_/workspace/inkline/CLAUDE.md: copy file error[Command '['docker', 'cp', 'tmp_630ddab0:/workspace/inkline/CLAUDE.md', '../data/octo_image_file/md-inkline-use-notification/AGENTS.md']' returned non-zero exit status 1.]


2026-07-06 20:25:30,793 - WARNING - Instance md-inkline-use-notification: Claude.md not found
2026-07-06 20:25:30,797 - WARNING - Instance md-inkline-use-notification: AGENTS.md not found
 99%|█████████▉| 141/142 [5:53:58<02:47, 167.43s/it]

[ERROR] minimaxai/feedfeed:md_inkline_tmp_b45c7bbe_/workspace/inkline/CLAUDE.md: copy file error[Command '['docker', 'cp', 'tmp_b45c7bbe:/workspace/inkline/CLAUDE.md', '../data/octo_image_file/md-inkline-clone-schema/Claude.md']' returned non-zero exit status 1.]
[ERROR] minimaxai/feedfeed:md_inkline_tmp_b45c7bbe_/workspace/inkline/CLAUDE.md: copy file error[Command '['docker', 'cp', 'tmp_b45c7bbe:/workspace/inkline/CLAUDE.md', '../data/octo_image_file/md-inkline-clone-schema/AGENTS.md']' returned non-zero exit status 1.]


2026-07-06 20:27:45,277 - WARNING - Instance md-inkline-clone-schema: Claude.md not found
2026-07-06 20:27:45,279 - WARNING - Instance md-inkline-clone-schema: AGENTS.md not found
100%|██████████| 142/142 [5:56:13<00:00, 150.52s/it]


In [ ]:
from constants.addr import load_json

instance_query_dict = load_json(INSTANCE_QUERY_ADDR)

Has_MD = {
    id: query
    for id,query in instance_query_dict.items()
    if query["Claude.md"] != "" or query["AGENTS.md"]!=""
}

print(json.dumps(Has_MD, indent = 4))
print(len(Has_MD))

{
    "md-aws-mcp-server-logging-over-print": {
        "user_query": [
            "Security Requirement: Enhance security validation before command execution.\n\nBackground: The current system's validation of user-input commands is not strict enough, posing potential security risks.\n\nValidation rules to implement:\n1. Commands must start with 'aws'\n2. Prohibit command injection characters (`;` `&&` `||` `$()` backticks, etc.)\n3. Pipe character `|` requires special handling: allowed but must verify that commands after the pipe are safe (only allow grep, sort, head, tail, wc, etc.)\n4. Detect and reject dangerous operations (commands related to rm, delete require additional confirmation)\n\nRequirements:\n- Return detailed error reasons when validation fails to help users correct their input\n- Log all validation failures for security auditing\n\nPlease implement the command_validator module."
        ],
        "system_prompt": "",
        "Claude.md": "# AWS MCP Server Developmen